# Daily data change ETL

You have been tasked with extracting a daily dump from a live database system.
The team in charge of the database has given you a function which generates an
export you can copy to your warehouse: from the `origin` folder to the
`destination` folder.

`origin/` is the database. Treat it as read-only. Your job is to fill
`destination/`.

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from generator import simulate_day, validate

ORIGIN = Path("origin")
DESTINATION = Path("destination")

# Start from an empty database and an empty warehouse.
shutil.rmtree(ORIGIN, ignore_errors=True)
shutil.rmtree(DESTINATION, ignore_errors=True)
DESTINATION.mkdir()

## One day of the database

Every call to `simulate_day()` moves the database forward by one day and writes
that day's dump to `origin/`.

In [ ]:
day = simulate_day(ORIGIN, seed=42)
print(day)
print(sorted(path.name for path in ORIGIN.iterdir()))
pd.read_csv(ORIGIN / f"{day}.csv").head()

## Hint: extracting a day

Getting the day's dump into your warehouse is a copy.

In [ ]:
shutil.copy(ORIGIN / f"{day}.csv", DESTINATION / f"{day}.csv");

## A week of extractions

Run the database and the extraction together for the rest of the week.

In [ ]:
for _ in range(6):
    day = simulate_day(ORIGIN, seed=42)
    shutil.copy(ORIGIN / f"{day}.csv", DESTINATION / f"{day}.csv")

sorted(path.name for path in DESTINATION.iterdir())

## Does the warehouse match the database?

In [ ]:
def amount_per_date(folder: Path) -> pd.Series:
    return pd.Series(
        {
            path.stem: pd.read_csv(path)["amount"].sum()
            for path in sorted(folder.glob("*.csv"))
        }
    )


comparison = pd.DataFrame(
    {"origin": amount_per_date(ORIGIN), "destination": amount_per_date(DESTINATION)}
)
ax = comparison.plot(marker="o", figsize=(9, 4), title="Total amount per date")
ax.set_ylabel("sum(amount)")
plt.show()

In [ ]:
validate(ORIGIN, DESTINATION);

## Your challenge

Why aren't they the same?

Write an extraction that keeps `destination/` the same as `origin/`. Run it
after every `simulate_day()`, and check it with `validate()`.